### Power BI Dashboard Development

Objective: Transform your project's validated datasets and analytical results into an interactive business intelligence dashboard that communicates product adoption, user engagement, retention, churn, A/B testing, and causal analysis.

### Step 1 — Audit Project Data and Prepare Power BI Sources

#### 1.1 — Set the project directory

In [1]:
from pathlib import Path

project_root = Path(
    r"C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics"
)

data_path = project_root / "data"
synthetic_path = data_path / "synthetic"
documentation_path = project_root / "python" / "documentation"

print("Project Directory:", project_root)
print("Project Exists:", project_root.exists())
print("Synthetic Data Exists:", synthetic_path.exists())
print("Documentation Exists:", documentation_path.exists())

Project Directory: C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics
Project Exists: True
Synthetic Data Exists: True
Documentation Exists: True


#### 1.2 — Inspect all project CSV files

In [2]:
import pandas as pd

csv_files = list(project_root.rglob("*.csv"))

file_inventory = []

for file in csv_files:
    try:
        df = pd.read_csv(file, nrows=5)

        file_inventory.append({
            "file_name": file.name,
            "folder": str(file.parent.relative_to(project_root)),
            "columns": len(df.columns),
            "column_names": ", ".join(df.columns)
        })

    except Exception as error:
        file_inventory.append({
            "file_name": file.name,
            "folder": str(file.parent.relative_to(project_root)),
            "columns": "Error",
            "column_names": str(error)
        })

inventory_df = pd.DataFrame(file_inventory)

print("Total CSV Files:", len(inventory_df))

display(inventory_df)

Total CSV Files: 106


,file_name,folder,columns,column_names
0,api_performance.csv,python\data,13,"endpoint, total_api_records, total_users, tota..."
1,api_performance_correlation_matrix.csv,python\data,12,"Unnamed: 0, total_api_records, total_users, to..."
2,api_performance_eda_report.csv,python\data,13,"endpoint, total_api_records, total_users, tota..."
3,business_api_performance_ranking.csv,python\data,13,"endpoint, total_api_records, total_users, tota..."
4,business_engagement_summary.csv,python\data,3,"engagement_segment, user_count, percentage"
...,...,...,...,...
101,feature_usage.csv,data\synthetic,6,"usage_id, user_id, feature_name, first_used_at..."
102,organizations.csv,data\synthetic,6,"organization_id, organization_name, industry, ..."
103,product_events.csv,data\synthetic,8,"event_id, user_id, event_name, event_timestamp..."
104,subscriptions.csv,data\synthetic,7,"subscription_id, organization_id, plan_type, s..."


#### 1.3 — Inspect existing documentation

In [3]:
documentation_files = list(
    documentation_path.glob("*")
)

print("Available Documentation Files:\n")

for file in documentation_files:
    print(file.name)

Available Documentation Files:

BUSINESS_INSIGHTS.md
CAUSAL_IMPACT_ANALYSIS_REPORT.md
MACHINE_LEARNING_REPORT.md
ML_METHODOLOGY.md


### Step 2 — Categorize CSV files

In [4]:
# Step 2: Categorize Project CSV Files

from collections import Counter

folder_counts = Counter(
    str(file.parent.relative_to(project_root))
    for file in csv_files
)

folder_summary = pd.DataFrame(
    folder_counts.items(),
    columns=["folder", "csv_file_count"]
).sort_values(
    by="csv_file_count",
    ascending=False
)

display(folder_summary)

print("\nSynthetic Dataset Files:")

synthetic_files = sorted(synthetic_path.glob("*.csv"))

for file in synthetic_files:
    print(file.name)

print("\nTotal Synthetic Dataset Files:", len(synthetic_files))

,folder,csv_file_count
4,python\notebooks\data\processed,36
0,python\data,20
5,dbt\devflow_analytics\.venv\Lib\site-packages\...,20
6,dbt\devflow_analytics\.venv\Lib\site-packages\...,10
7,data\synthetic,8
1,python\notebooks\data,5
3,python\notebooks\models\interpretation,4
2,python\notebooks\models\evaluation,3



Synthetic Dataset Files:
api_usage.csv
experiment_assignments.csv
experiment_outcomes.csv
feature_usage.csv
organizations.csv
product_events.csv
subscriptions.csv
users.csv

Total Synthetic Dataset Files: 8


### Step 3 — Identify Power BI candidate datasets

We will inspect three locations separately:
1. data\synthetic — original source datasets.
2. python\notebooks\data\processed — processed datasets.
3. python\data — analytical summaries and KPI outputs.

In [5]:
# Step 3: Identify Power BI Candidate Datasets

source_folders = {
    "Synthetic": project_root / "data" / "synthetic",
    "Processed": project_root / "python" / "notebooks" / "data" / "processed",
    "Analytics": project_root / "python" / "data"
}

candidate_files = []

for category, folder in source_folders.items():

    for file in sorted(folder.glob("*.csv")):

        try:
            df = pd.read_csv(file, nrows=5)

            candidate_files.append({
                "category": category,
                "file_name": file.name,
                "columns": len(df.columns),
                "column_names": ", ".join(df.columns)
            })

        except Exception as error:

            candidate_files.append({
                "category": category,
                "file_name": file.name,
                "columns": "Error",
                "column_names": str(error)
            })

candidate_inventory = pd.DataFrame(candidate_files)

print("Total Candidate CSV Files:", len(candidate_inventory))

display(candidate_inventory)

Total Candidate CSV Files: 64


,category,file_name,columns,column_names
0,Synthetic,api_usage.csv,8,"api_usage_id, user_id, request_timestamp, endp..."
1,Synthetic,experiment_assignments.csv,5,"assignment_id, experiment_id, user_id, variant..."
2,Synthetic,experiment_outcomes.csv,8,"outcome_id, assignment_id, user_id, sdk_integr..."
3,Synthetic,feature_usage.csv,6,"usage_id, user_id, feature_name, first_used_at..."
4,Synthetic,organizations.csv,6,"organization_id, organization_name, industry, ..."
...,...,...,...,...
59,Analytics,user_engagement.csv,12,"user_id, active_days, total_events, total_sess..."
60,Analytics,user_engagement_activity_summary.csv,6,"Unnamed: 0, mean, median, min, max, std"
61,Analytics,user_engagement_correlation_matrix.csv,10,"Unnamed: 0, user_id, active_days, total_events..."
62,Analytics,user_engagement_correlation_report.csv,4,"Metric_1, Metric_2, Correlation, Absolute_Corr..."


### Step 4 — Display the complete candidate inventory

In [6]:
# Step 4: Display Complete Candidate Inventory

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

display(
    candidate_inventory.sort_values(
        by=["category", "file_name"]
    ).reset_index(drop=True)
)

,category,file_name,columns,column_names
0,Analytics,api_performance.csv,13,"endpoint, total_api_records, total_users, total_requests, successful_requests, failed_requests, error_rate_percentage, success_rate_percentage, average_response_time_ms, maximum_response_time_ms, error_rate_rank, response_time_rank, api_health_status"
1,Analytics,api_performance_correlation_matrix.csv,12,"Unnamed: 0, total_api_records, total_users, total_requests, successful_requests, failed_requests, error_rate_percentage, success_rate_percentage, average_response_time_ms, maximum_response_time_ms, error_rate_rank, response_time_rank"
2,Analytics,api_performance_eda_report.csv,13,"endpoint, total_api_records, total_users, total_requests, successful_requests, failed_requests, error_rate_percentage, success_rate_percentage, average_response_time_ms, maximum_response_time_ms, error_rate_rank, response_time_rank, api_health_status"
3,Analytics,business_api_performance_ranking.csv,13,"endpoint, total_api_records, total_users, total_requests, successful_requests, failed_requests, error_rate_percentage, success_rate_percentage, average_response_time_ms, maximum_response_time_ms, error_rate_rank, response_time_rank, api_health_status"
4,Analytics,business_engagement_summary.csv,3,"engagement_segment, user_count, percentage"
5,Analytics,business_insights_dataset_summary.csv,5,"Dataset, Rows, Columns, Missing Values, Duplicate Rows"
6,Analytics,business_recommendations.csv,4,"Area, Business Question, Recommended Action, Priority"
7,Analytics,churn_analysis.csv,12,"plan_type, total_subscriptions, total_organizations, churned_subscriptions, non_churned_subscriptions, churn_rate_percentage, non_churn_rate_percentage, total_monthly_subscription_value, average_monthly_price, first_subscription_date, latest_subscription_date, churn_risk_rank"
8,Analytics,data_quality_report.csv,5,"Dataset, Total Rows, Total Columns, Missing Values, Duplicate Rows"
9,Analytics,dataset_overview_report.csv,6,"Dataset, Rows, Columns, Missing Values, Duplicate Rows, Memory Usage (KB)"


In [7]:
# List Processed and Analytics CSV Files

for category in ["Processed", "Analytics"]:

    print(f"\n{'=' * 60}")
    print(f"{category.upper()} DATASETS")
    print("=" * 60)

    files = candidate_inventory.loc[
        candidate_inventory["category"] == category,
        "file_name"
    ].tolist()

    for number, filename in enumerate(files, start=1):
        print(f"{number}. {filename}")

    print("Total:", len(files))


PROCESSED DATASETS
1. api_monitoring_adoption_summary.csv
2. api_usage_features.csv
3. baseline_model_comparison.csv
4. categorical_value_report.csv
5. data_type_report.csv
6. date_type_report.csv
7. decision_tree_results.csv
8. dummy_classifier_results.csv
9. duplicate_summary.csv
10. feature_engineered_dataset.csv
11. feature_engineering_date_validation.csv
12. final_test_predictions.csv
13. historical_user_cohort_dataset.csv
14. logistic_regression_results.csv
15. missing_value_report.csv
16. missing_value_summary.csv
17. ml_eda_api_adoption_comparison.csv
18. ml_eda_dataset_summary.csv
19. ml_eda_engagement_adoption_comparison.csv
20. ml_eda_feature_distribution.csv
21. ml_eda_subscription_distribution.csv
22. ml_eda_user_api_activity.csv
23. ml_eda_user_engagement.csv
24. model_predictors.csv
25. model_target.csv
26. numerical_outlier_report.csv
27. preprocessing_summary.csv
28. primary_key_duplicate_report.csv
29. product_event_features.csv
30. random_forest_results.csv
31. test

### Step 4 — Select Power BI data sources

#### Inspect selected dataset quality

In [8]:
# Step 4: Inspect Selected Power BI Datasets

import pandas as pd

# Core source datasets
synthetic_names = [
    "users.csv",
    "organizations.csv",
    "product_events.csv",
    "feature_usage.csv",
    "api_usage.csv",
    "subscriptions.csv",
    "experiment_assignments.csv",
    "experiment_outcomes.csv"
]

# Analytical datasets
analytics_names = [
    "api_performance.csv",
    "churn_analysis.csv",
    "experiment_analysis.csv",
    "feature_adoption.csv",
    "user_engagement.csv",
    "business_engagement_summary.csv",
    "business_recommendations.csv"
]

selected_files = {
    "Synthetic": [
        synthetic_path / name
        for name in synthetic_names
    ],
    "Analytics": [
        project_root / "python" / "data" / name
        for name in analytics_names
    ]
}

dataset_quality = []

for category, files in selected_files.items():

    for file in files:

        df = pd.read_csv(file)

        dataset_quality.append({
            "category": category,
            "file_name": file.name,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_values": int(df.isna().sum().sum()),
            "duplicate_rows": int(df.duplicated().sum()),
            "memory_mb": round(
                df.memory_usage(deep=True).sum() / (1024 ** 2), 3
            )
        })

quality_df = pd.DataFrame(dataset_quality)

display(quality_df)

print("\nTotal Selected Datasets:", len(quality_df))
print("Total Rows:", quality_df["rows"].sum())
print("Total Missing Values:", quality_df["missing_values"].sum())
print("Total Duplicate Rows:", quality_df["duplicate_rows"].sum())

,category,file_name,rows,columns,missing_values,duplicate_rows,memory_mb
0,Synthetic,users.csv,10000,7,0,0,0.856
1,Synthetic,organizations.csv,1500,6,0,0,0.137
2,Synthetic,product_events.csv,80000,8,0,0,9.487
3,Synthetic,feature_usage.csv,20041,6,0,0,1.584
4,Synthetic,api_usage.csv,50000,8,0,0,4.228
5,Synthetic,subscriptions.csv,1500,7,1043,0,0.116
6,Synthetic,experiment_assignments.csv,6000,5,0,0,0.481
7,Synthetic,experiment_outcomes.csv,6000,8,0,0,0.424
8,Analytics,api_performance.csv,6,13,0,0,0.001
9,Analytics,churn_analysis.csv,3,12,0,0,0.000



Total Selected Datasets: 15
Total Rows: 185067
Total Missing Values: 1118
Total Duplicate Rows: 0


#### Inspect dataset columns

In [9]:
# Inspect the columns of every selected dataset

for category, files in selected_files.items():

    print(f"\n{'=' * 65}")
    print(category.upper())
    print("=" * 65)

    for file in files:

        df = pd.read_csv(file, nrows=5)

        print(f"\nDataset: {file.name}")
        print("Columns:", list(df.columns))


SYNTHETIC

Dataset: users.csv
Columns: ['user_id', 'organization_id', 'signup_date', 'country', 'user_role', 'acquisition_channel', 'is_verified']

Dataset: organizations.csv
Columns: ['organization_id', 'organization_name', 'industry', 'company_size', 'created_date', 'country']

Dataset: product_events.csv
Columns: ['event_id', 'user_id', 'event_name', 'event_timestamp', 'session_id', 'platform', 'device_type', 'event_properties']

Dataset: feature_usage.csv
Columns: ['usage_id', 'user_id', 'feature_name', 'first_used_at', 'usage_count', 'last_used_at']

Dataset: api_usage.csv
Columns: ['api_usage_id', 'user_id', 'request_timestamp', 'endpoint', 'request_count', 'success_count', 'error_count', 'response_time_ms']

Dataset: subscriptions.csv
Columns: ['subscription_id', 'organization_id', 'plan_type', 'subscription_status', 'monthly_price', 'start_date', 'end_date']

Dataset: experiment_assignments.csv
Columns: ['assignment_id', 'experiment_id', 'user_id', 'variant', 'assigned_at']

D

#### Step 5 — Detailed missing-value investigation

#### 5.1 — Inspect missing values in subscriptions

In [10]:
# Step 5.1: Investigate Subscription Missing Values

subscriptions = pd.read_csv(
    synthetic_path / "subscriptions.csv"
)

print("Subscription Status Distribution:")
display(
    subscriptions["subscription_status"].value_counts(
        dropna=False
    )
)

print("\nMissing Values by Column:")
subscription_missing = (
    subscriptions.isna().sum()
    .to_frame("missing_count")
)

subscription_missing["missing_percentage"] = (
    subscription_missing["missing_count"]
    / len(subscriptions) * 100
)

display(subscription_missing.round(2))

print("\nMissing End Dates by Subscription Status:")

display(
    subscriptions.groupby(
        "subscription_status",
        dropna=False
    )["end_date"]
    .agg(
        total_records="size",
        missing_end_dates=lambda x: x.isna().sum()
    )
)

Subscription Status Distribution:


subscription_status
Active       1043
Cancelled     241
Expired       216
Name: count, dtype: int64


Missing Values by Column:


,missing_count,missing_percentage
subscription_id,0,0.00
organization_id,0,0.00
plan_type,0,0.00
subscription_status,0,0.00
monthly_price,0,0.00
start_date,0,0.00
end_date,1043,69.53



Missing End Dates by Subscription Status:


,total_records,missing_end_dates
subscription_status,,
Active,1043,1043
Cancelled,241,0
Expired,216,0


#### 5.2 — Inspect missing values in user engagement

In [11]:
# Step 5.2: Investigate User Engagement Missing Values

user_engagement = pd.read_csv(
    project_root / "python" / "data" / "user_engagement.csv"
)

engagement_missing = (
    user_engagement.isna().sum()
    .to_frame("missing_count")
)

engagement_missing["missing_percentage"] = (
    engagement_missing["missing_count"]
    / len(user_engagement) * 100
)

engagement_missing = engagement_missing[
    engagement_missing["missing_count"] > 0
].sort_values(
    by="missing_count",
    ascending=False
)

print("Missing Values in User Engagement:")
display(engagement_missing.round(2))

print("\nSample Rows Containing Missing Values:")

display(
    user_engagement[
        user_engagement.isna().any(axis=1)
    ].head(10)
)

Missing Values in User Engagement:


,missing_count,missing_percentage
api_success_rate_percentage,75,0.75



Sample Rows Containing Missing Values:


,user_id,active_days,total_events,total_sessions,total_api_requests,successful_api_requests,failed_api_requests,api_success_rate_percentage,first_activity_date,last_activity_date,activity_lifespan_days,engagement_segment
0,2993,1,1,1,0,0,0,NaN,2026-01-07,2026-01-07,0,Very Low Engagement
1,6905,1,1,1,0,0,0,NaN,2024-12-18,2024-12-18,0,Very Low Engagement
2,3503,1,1,1,0,0,0,NaN,2024-08-02,2024-08-02,0,Very Low Engagement
3,9846,1,1,1,0,0,0,NaN,2025-02-13,2025-02-13,0,Very Low Engagement
5,7516,1,1,1,0,0,0,NaN,2024-09-28,2024-09-28,0,Very Low Engagement
6,8787,1,1,1,0,0,0,NaN,2025-10-13,2025-10-13,0,Very Low Engagement
7,4129,1,1,1,0,0,0,NaN,2026-02-10,2026-02-10,0,Very Low Engagement
8,7782,1,1,1,0,0,0,NaN,2024-06-06,2024-06-06,0,Very Low Engagement
9,9723,1,1,1,0,0,0,NaN,2024-12-05,2024-12-05,0,Very Low Engagement
10,6109,1,1,1,0,0,0,NaN,2025-11-07,2025-11-07,0,Very Low Engagement


#### 5.3 — Validate duplicate records

In [12]:
# Step 5.3: Validate Duplicate Records

for category, files in selected_files.items():

    for file in files:

        df = pd.read_csv(file)

        print(
            f"{file.name}: "
            f"{df.duplicated().sum()} duplicate rows"
        )

users.csv: 0 duplicate rows
organizations.csv: 0 duplicate rows
product_events.csv: 0 duplicate rows
feature_usage.csv: 0 duplicate rows
api_usage.csv: 0 duplicate rows
subscriptions.csv: 0 duplicate rows
experiment_assignments.csv: 0 duplicate rows
experiment_outcomes.csv: 0 duplicate rows
api_performance.csv: 0 duplicate rows
churn_analysis.csv: 0 duplicate rows
experiment_analysis.csv: 0 duplicate rows
feature_adoption.csv: 0 duplicate rows
user_engagement.csv: 0 duplicate rows
business_engagement_summary.csv: 0 duplicate rows
business_recommendations.csv: 0 duplicate rows


### Step 6 — Prepare Power BI reporting datasets

#### 6.1 — Create the Power BI data directory

In [13]:
# Step 6.1: Create Power BI Data Directory

power_bi_path = project_root / "power_bi"
power_bi_data_path = power_bi_path / "data"

power_bi_data_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Power BI Directory:", power_bi_path)
print("Power BI Data Directory:", power_bi_data_path)
print("Directory Exists:", power_bi_data_path.exists())

Power BI Directory: C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\power_bi
Power BI Data Directory: C:\Users\User\Documents\AI-Feature-Adoption-Churn-Analytics\power_bi\data
Directory Exists: True


#### 6.2 — Export selected datasets

In [14]:
# Step 6.2: Export Selected Datasets for Power BI

export_names = {
    "users.csv": synthetic_path / "users.csv",
    "organizations.csv": synthetic_path / "organizations.csv",
    "product_events.csv": synthetic_path / "product_events.csv",
    "feature_usage.csv": synthetic_path / "feature_usage.csv",
    "api_usage.csv": synthetic_path / "api_usage.csv",
    "subscriptions.csv": synthetic_path / "subscriptions.csv",
    "experiment_assignments.csv": synthetic_path / "experiment_assignments.csv",
    "experiment_outcomes.csv": synthetic_path / "experiment_outcomes.csv",
    "api_performance.csv": project_root / "python" / "data" / "api_performance.csv",
    "churn_analysis.csv": project_root / "python" / "data" / "churn_analysis.csv",
    "experiment_analysis.csv": project_root / "python" / "data" / "experiment_analysis.csv",
    "feature_adoption.csv": project_root / "python" / "data" / "feature_adoption.csv",
    "user_engagement.csv": project_root / "python" / "data" / "user_engagement.csv",
    "business_engagement_summary.csv": project_root / "python" / "data" / "business_engagement_summary.csv"
}

export_summary = []

for output_name, source_file in export_names.items():

    df = pd.read_csv(source_file)

    output_file = power_bi_data_path / output_name

    df.to_csv(
        output_file,
        index=False
    )

    export_summary.append({
        "file_name": output_name,
        "rows": len(df),
        "columns": len(df.columns),
        "file_exists": output_file.exists()
    })

export_summary_df = pd.DataFrame(export_summary)

display(export_summary_df)

print("\nTotal Exported Datasets:", len(export_summary_df))
print("All Files Exported:", export_summary_df["file_exists"].all())

,file_name,rows,columns,file_exists
0,users.csv,10000,7,True
1,organizations.csv,1500,6,True
2,product_events.csv,80000,8,True
3,feature_usage.csv,20041,6,True
4,api_usage.csv,50000,8,True
5,subscriptions.csv,1500,7,True
6,experiment_assignments.csv,6000,5,True
7,experiment_outcomes.csv,6000,8,True
8,api_performance.csv,6,13,True
9,churn_analysis.csv,3,12,True



Total Exported Datasets: 14
All Files Exported: True


#### 6.3 — Verify exported datasets

In [15]:
# Step 6.3: Verify Power BI Export Files

exported_files = sorted(
    power_bi_data_path.glob("*.csv")
)

print("Exported CSV Files:\n")

for file in exported_files:
    print(file.name)

print("\nTotal Exported Files:", len(exported_files))

Exported CSV Files:

api_performance.csv
api_usage.csv
business_engagement_summary.csv
churn_analysis.csv
experiment_analysis.csv
experiment_assignments.csv
experiment_outcomes.csv
feature_adoption.csv
feature_usage.csv
organizations.csv
product_events.csv
subscriptions.csv
user_engagement.csv
users.csv

Total Exported Files: 14


### Step 7 — Validate Primary Keys and Foreign Keys

Objective: Check whether the selected datasets have unique identifiers and valid relationships before importing them into Power BI.

#### Step 7.1 — Load the exported datasets

In [16]:

# Step 7.1: Load Exported Datasets for Relationship Validation

power_bi_data = {}

for file in exported_files:
    df_name = file.stem
    power_bi_data[df_name] = pd.read_csv(file)

print("Datasets Loaded:", len(power_bi_data))

for name, df in power_bi_data.items():
    print(f"{name}: {df.shape}")


Datasets Loaded: 14
api_performance: (6, 13)
api_usage: (50000, 8)
business_engagement_summary: (4, 3)
churn_analysis: (3, 12)
experiment_analysis: (2, 11)
experiment_assignments: (6000, 5)
experiment_outcomes: (6000, 8)
feature_adoption: (7, 10)
feature_usage: (20041, 6)
organizations: (1500, 6)
product_events: (80000, 8)
subscriptions: (1500, 7)
user_engagement: (9999, 12)
users: (10000, 7)


#### Step 7.2 — Check primary key uniqueness

In [17]:

# Step 7.2: Validate Primary Key Uniqueness

primary_keys = {
    "users": "user_id",
    "organizations": "organization_id",
    "product_events": "event_id",
    "feature_usage": "usage_id",
    "api_usage": "api_usage_id",
    "subscriptions": "subscription_id",
    "experiment_assignments": "assignment_id",
    "experiment_outcomes": "outcome_id"
}

primary_key_results = []

for table, key in primary_keys.items():
    df = power_bi_data[table]

    primary_key_results.append({
        "table": table,
        "primary_key": key,
        "total_rows": len(df),
        "missing_keys": df[key].isna().sum(),
        "unique_keys": df[key].nunique(),
        "duplicate_keys": df[key].duplicated().sum(),
        "primary_key_valid": (
            df[key].notna().all() and
            df[key].is_unique
        )
    })

primary_key_summary = pd.DataFrame(primary_key_results)

display(primary_key_summary)


,table,primary_key,total_rows,missing_keys,unique_keys,duplicate_keys,primary_key_valid
0,users,user_id,10000,0,10000,0,True
1,organizations,organization_id,1500,0,1500,0,True
2,product_events,event_id,80000,0,80000,0,True
3,feature_usage,usage_id,20041,0,20041,0,True
4,api_usage,api_usage_id,50000,0,50000,0,True
5,subscriptions,subscription_id,1500,0,1500,0,True
6,experiment_assignments,assignment_id,6000,0,6000,0,True
7,experiment_outcomes,outcome_id,6000,0,6000,0,True


### Step 8 — Validate Foreign Keys

Objective: Confirm that child-table identifiers match records in their parent tables.

#### Step 8.1 — Define expected relationships

In [18]:

# Step 8.1: Define Expected Foreign Key Relationships

foreign_key_relationships = [
    {
        "child_table": "users",
        "child_key": "organization_id",
        "parent_table": "organizations",
        "parent_key": "organization_id"
    },
    {
        "child_table": "product_events",
        "child_key": "user_id",
        "parent_table": "users",
        "parent_key": "user_id"
    },
    {
        "child_table": "feature_usage",
        "child_key": "user_id",
        "parent_table": "users",
        "parent_key": "user_id"
    },
    {
        "child_table": "api_usage",
        "child_key": "user_id",
        "parent_table": "users",
        "parent_key": "user_id"
    },
    {
        "child_table": "subscriptions",
        "child_key": "organization_id",
        "parent_table": "organizations",
        "parent_key": "organization_id"
    },
    {
        "child_table": "experiment_assignments",
        "child_key": "user_id",
        "parent_table": "users",
        "parent_key": "user_id"
    },
    {
        "child_table": "experiment_outcomes",
        "child_key": "assignment_id",
        "parent_table": "experiment_assignments",
        "parent_key": "assignment_id"
    },
    {
        "child_table": "experiment_outcomes",
        "child_key": "user_id",
        "parent_table": "users",
        "parent_key": "user_id"
    }
]

print("Expected Foreign Key Relationships:")
print("Total Relationships:", len(foreign_key_relationships))

for relationship in foreign_key_relationships:
    print(
        f'{relationship["child_table"]}.{relationship["child_key"]} '
        f'-> '
        f'{relationship["parent_table"]}.{relationship["parent_key"]}'
    )


Expected Foreign Key Relationships:
Total Relationships: 8
users.organization_id -> organizations.organization_id
product_events.user_id -> users.user_id
feature_usage.user_id -> users.user_id
api_usage.user_id -> users.user_id
subscriptions.organization_id -> organizations.organization_id
experiment_assignments.user_id -> users.user_id
experiment_outcomes.assignment_id -> experiment_assignments.assignment_id
experiment_outcomes.user_id -> users.user_id


#### Step 8.2 — Check referential integrity

In [19]:

# Step 8.2: Validate Foreign Key References

foreign_key_results = []

for relationship in foreign_key_relationships:

    child_df = power_bi_data[relationship["child_table"]]
    parent_df = power_bi_data[relationship["parent_table"]]

    child_key = relationship["child_key"]
    parent_key = relationship["parent_key"]

    child_values = child_df[child_key]
    parent_values = set(parent_df[parent_key].dropna())

    missing_count = child_values.isna().sum()

    unmatched_count = (
        child_values.notna() &
        ~child_values.isin(parent_values)
    ).sum()

    foreign_key_results.append({
        "child_table": relationship["child_table"],
        "child_key": child_key,
        "parent_table": relationship["parent_table"],
        "parent_key": parent_key,
        "total_rows": len(child_df),
        "missing_foreign_keys": missing_count,
        "unmatched_foreign_keys": unmatched_count,
        "relationship_valid": (
            missing_count == 0 and unmatched_count == 0
        )
    })

foreign_key_summary = pd.DataFrame(foreign_key_results)

display(foreign_key_summary)


,child_table,child_key,parent_table,parent_key,total_rows,missing_foreign_keys,unmatched_foreign_keys,relationship_valid
0,users,organization_id,organizations,organization_id,10000,0,0,True
1,product_events,user_id,users,user_id,80000,0,0,True
2,feature_usage,user_id,users,user_id,20041,0,0,True
3,api_usage,user_id,users,user_id,50000,0,0,True
4,subscriptions,organization_id,organizations,organization_id,1500,0,0,True
5,experiment_assignments,user_id,users,user_id,6000,0,0,True
6,experiment_outcomes,assignment_id,experiment_assignments,assignment_id,6000,0,0,True
7,experiment_outcomes,user_id,users,user_id,6000,0,0,True


### Step 9 — Validate Experiment User Consistency

Objective: Confirm that each experiment outcome belongs to the same user recorded in its assignment.

#### Step 9.1 — Compare assignment and outcome users

In [20]:

# Step 9.1: Validate Experiment Assignment and Outcome User Consistency

assignments_df = power_bi_data["experiment_assignments"]
outcomes_df = power_bi_data["experiment_outcomes"]

experiment_consistency = outcomes_df.merge(
    assignments_df[["assignment_id", "user_id"]],
    on="assignment_id",
    how="left",
    suffixes=("_outcome", "_assignment"),
    validate="one_to_one"
)

experiment_consistency["user_id_match"] = (
    experiment_consistency["user_id_outcome"] ==
    experiment_consistency["user_id_assignment"]
)

print("Total Experiment Outcomes:", len(experiment_consistency))
print(
    "Matching User IDs:",
    experiment_consistency["user_id_match"].sum()
)
print(
    "Mismatching User IDs:",
    (~experiment_consistency["user_id_match"]).sum()
)

display(
    experiment_consistency.loc[
        ~experiment_consistency["user_id_match"],
        [
            "assignment_id",
            "user_id_outcome",
            "user_id_assignment"
        ]
    ].head(10)
)


Total Experiment Outcomes: 6000
Matching User IDs: 6000
Mismatching User IDs: 0


,assignment_id,user_id_outcome,user_id_assignment


#### Step 9.2 — Validate assignment uniqueness per user

In [21]:

# Step 9.2: Check Multiple Experiment Assignments per User

assignment_counts = (
    assignments_df
    .groupby("user_id")
    .size()
    .reset_index(name="assignment_count")
)

multiple_assignments = assignment_counts[
    assignment_counts["assignment_count"] > 1
]

print("Total Assigned Users:", assignments_df["user_id"].nunique())
print("Total Assignment Records:", len(assignments_df))
print("Users with Multiple Assignments:", len(multiple_assignments))

display(multiple_assignments.head(10))


Total Assigned Users: 6000
Total Assignment Records: 6000
Users with Multiple Assignments: 0


,user_id,assignment_count


### Step 10 — Design the Power BI Data Model

Objective: Define the table relationships before creating them in Power BI Desktop.

We will use a star-schema-oriented design, with users and organizations as descriptive tables and activity/experiment tables as transactional tables.

#### Step 10.1— Create a relationship specification table

In [22]:

# Step 10.1: Create Power BI Relationship Specification

relationship_spec = pd.DataFrame([
    ["organizations", "organization_id", "users", "organization_id", "One-to-Many"],
    ["users", "user_id", "product_events", "user_id", "One-to-Many"],
    ["users", "user_id", "feature_usage", "user_id", "One-to-Many"],
    ["users", "user_id", "api_usage", "user_id", "One-to-Many"],
    ["organizations", "organization_id", "subscriptions", "organization_id", "One-to-Many"],
    ["users", "user_id", "experiment_assignments", "user_id", "One-to-Many"],
    ["experiment_assignments", "assignment_id", "experiment_outcomes", "assignment_id", "One-to-One"]
], columns=[
    "parent_table",
    "parent_key",
    "child_table",
    "child_key",
    "expected_cardinality"
])

display(relationship_spec)


,parent_table,parent_key,child_table,child_key,expected_cardinality
0,organizations,organization_id,users,organization_id,One-to-Many
1,users,user_id,product_events,user_id,One-to-Many
2,users,user_id,feature_usage,user_id,One-to-Many
3,users,user_id,api_usage,user_id,One-to-Many
4,organizations,organization_id,subscriptions,organization_id,One-to-Many
5,users,user_id,experiment_assignments,user_id,One-to-Many
6,experiment_assignments,assignment_id,experiment_outcomes,assignment_id,One-to-One


### Step 11 — Import CSV Files into Power BI Desktop

Objective: Load all 14 validated reporting datasets into Power BI.

#### Validate user_engagement

In [23]:

# Step 12.1: Validate User Engagement Relationship

users_df = power_bi_data["users"]
engagement_df = power_bi_data["user_engagement"]

print("Users rows:", len(users_df))
print("Engagement rows:", len(engagement_df))

print("\nUsers unique IDs:", users_df["user_id"].nunique())
print("Engagement unique IDs:", engagement_df["user_id"].nunique())

print("\nEngagement missing user IDs:", engagement_df["user_id"].isna().sum())
print("Engagement duplicate user IDs:", engagement_df["user_id"].duplicated().sum())

unmatched_users = set(engagement_df["user_id"].dropna()) - set(users_df["user_id"].dropna())

users_without_engagement = set(users_df["user_id"].dropna()) - set(engagement_df["user_id"].dropna())

print("\nEngagement users not found in users table:", len(unmatched_users))
print("Users without engagement records:", len(users_without_engagement))


Users rows: 10000
Engagement rows: 9999

Users unique IDs: 10000
Engagement unique IDs: 9999

Engagement missing user IDs: 0
Engagement duplicate user IDs: 0

Engagement users not found in users table: 0
Users without engagement records: 1


In [24]:

# Step 12.1: Validate User Engagement Relationship

users_df = power_bi_data["users"]
engagement_df = power_bi_data["user_engagement"]

print("Users rows:", len(users_df))
print("Engagement rows:", len(engagement_df))

print("\nUsers unique IDs:", users_df["user_id"].nunique())
print("Engagement unique IDs:", engagement_df["user_id"].nunique())

print("\nEngagement missing user IDs:", engagement_df["user_id"].isna().sum())
print("Engagement duplicate user IDs:", engagement_df["user_id"].duplicated().sum())

unmatched_users = (
    set(engagement_df["user_id"].dropna())
    - set(users_df["user_id"].dropna())
)

users_without_engagement = (
    set(users_df["user_id"].dropna())
    - set(engagement_df["user_id"].dropna())
)

print("\nEngagement users not found in users table:", len(unmatched_users))
print("Users without engagement records:", len(users_without_engagement))


Users rows: 10000
Engagement rows: 9999

Users unique IDs: 10000
Engagement unique IDs: 9999

Engagement missing user IDs: 0
Engagement duplicate user IDs: 0

Engagement users not found in users table: 0
Users without engagement records: 1


### 3. Step 12.2 — Identify the user without an engagement record

In [25]:

# Step 12.2: Identify User Without Engagement Record

users_without_engagement_ids = (
    set(users_df["user_id"])
    - set(engagement_df["user_id"])
)

display(
    users_df[
        users_df["user_id"].isin(users_without_engagement_ids)
    ]
)


,user_id,organization_id,signup_date,country,user_role,acquisition_channel,is_verified
9731,9732,1114,2025-06-02,Singapore,Admin,Social Media,1


### Step 12.3 — Check the user's activity in source datasets

In [26]:

# Step 12.3: Check Activity for User Without Engagement Record

missing_engagement_user_id = 9732

activity_checks = {
    "product_events": power_bi_data["product_events"],
    "feature_usage": power_bi_data["feature_usage"],
    "api_usage": power_bi_data["api_usage"]
}

for table_name, df in activity_checks.items():
    activity_count = (
        df["user_id"] == missing_engagement_user_id
    ).sum()

    print(f"{table_name}: {activity_count} records")


product_events: 0 records
feature_usage: 1 records
api_usage: 0 records


### Step 12.4 — Inspect the feature usage record

In [27]:

# Step 12.4: Inspect Feature Usage for the Missing Engagement User

missing_engagement_user_id = 9732

user_feature_record = power_bi_data["feature_usage"].loc[
    power_bi_data["feature_usage"]["user_id"]
    == missing_engagement_user_id
]

display(user_feature_record)


,usage_id,user_id,feature_name,first_used_at,usage_count,last_used_at
19503,19504,9732,API Monitoring,2025-07-01,89,2025-07-01


### Step 12.5 — Verify the actual record

In [28]:
# Step 12.5: Verify Feature Usage Record

pd.set_option("display.max_columns", None)

display(
    power_bi_data["feature_usage"]
    .loc[
        power_bi_data["feature_usage"]["user_id"] == 9732
    ]
    .to_dict(orient="records")
)

[{'usage_id': 19504,
  'user_id': 9732,
  'feature_name': 'API Monitoring',
  'first_used_at': '2025-07-01',
  'usage_count': 89,
  'last_used_at': '2025-07-01'}]